# Lesson 03: D3PM from Scratch

We build a complete Discrete Denoising Diffusion Probabilistic Model: a transformer denoiser, the VLB training loss, and sampling. We train on a small text dataset and generate samples.

**Paper reference:** D3PM (Austin et al., 2021) Section 4.

In [ ]:
%pip install torch numpy matplotlib datasets tqdm --quiet

In [ ]:
import sys
sys.path.insert(0, '../../..')

import torch
from torch.utils.data import DataLoader
from shared.utils.seed import set_seed
from shared.utils.device import get_device
from shared.datasets.text import SimpleTokenizer, TextDataset

set_seed(42)
device = get_device()
print(f"Using device: {device}")

## 1. Prepare Data

We use a small collection of simple sentences for fast iteration. In the lab, you will use TinyStories.

In [ ]:
# Simple training data for fast iteration
texts = [
    "the cat sat on the mat",
    "a dog ran in the park",
    "the sun is bright today",
    "birds fly in the sky",
    "fish swim in the sea",
    "the moon shines at night",
    "rain falls from the clouds",
    "flowers grow in the garden",
    "children play in the yard",
    "the wind blows through trees",
] * 100  # Repeat for more training data

tokenizer = SimpleTokenizer(texts, level="char")
dataset = TextDataset(texts, tokenizer, seq_len=32)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Dataset size: {len(dataset)}")
print(f"Mask token ID: {tokenizer.mask_id}")
print(f"Sample: {tokenizer.decode(dataset[0].tolist())}")

## 2. Build the D3PM Model

In [ ]:
from src.d3pm_model import D3PMDenoiser, D3PM

# Small model for fast training
denoiser = D3PMDenoiser(
    vocab_size=tokenizer.vocab_size,
    d_model=64,
    n_heads=4,
    n_layers=3,
    max_seq_len=32,
    dropout=0.1,
).to(device)

d3pm = D3PM(
    denoiser=denoiser,
    vocab_size=tokenizer.vocab_size,
    num_timesteps=100,
    schedule="absorbing",
    mask_token_id=tokenizer.mask_id,
    hybrid_loss_coeff=0.01,
    device=device,
)

n_params = sum(p.numel() for p in denoiser.parameters())
print(f"Model parameters: {n_params:,}")

## 3. Training Loop

In [ ]:
from tqdm import tqdm

optimizer = torch.optim.Adam(denoiser.parameters(), lr=3e-4)
num_epochs = 20

losses = []
for epoch in range(num_epochs):
    epoch_loss = 0.0
    n_batches = 0
    
    for batch in dataloader:
        batch = batch.to(device)
        
        loss = d3pm.train_loss(batch)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(denoiser.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('D3PM Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Generate Samples

Let's sample from the trained model by running the reverse diffusion chain.

In [ ]:
# Generate samples
print("Generating samples...")
samples = d3pm.sample(batch_size=5, seq_len=32, temperature=0.8, verbose=True)

print("\nGenerated text:")
for i, sample in enumerate(samples):
    text = tokenizer.decode(sample.cpu().tolist())
    print(f"  [{i+1}] '{text}'")

In [ ]:
# Compare different temperatures
for temp in [0.3, 0.7, 1.0, 1.5]:
    print(f"\nTemperature = {temp}:")
    samples = d3pm.sample(batch_size=3, seq_len=32, temperature=temp)
    for sample in samples:
        text = tokenizer.decode(sample.cpu().tolist())
        print(f"  '{text}'")

## 5. Visualize the Denoising Process

Watch a single sample being progressively denoised.

In [ ]:
import torch.nn.functional as F

# Manual sampling with visualization
seq_len = 32
x_t = torch.full((1, seq_len), tokenizer.mask_id, dtype=torch.long, device=device)

snapshots = []
snapshot_times = [d3pm.num_timesteps, 75, 50, 25, 10, 5, 1]

with torch.no_grad():
    for t_val in range(d3pm.num_timesteps, 0, -1):
        if t_val in snapshot_times:
            text = tokenizer.decode(x_t[0].cpu().tolist())
            snapshots.append((t_val, text))
        
        t = torch.full((1,), t_val, device=device)
        logits = denoiser(x_t, t)
        x_0_probs = F.softmax(logits / 0.8, dim=-1)
        
        if t_val == 1:
            x_t = logits.argmax(dim=-1)
        else:
            x_t = d3pm._sample_reverse_step(x_t, x_0_probs, t_val)

# Add final result
text = tokenizer.decode(x_t[0].cpu().tolist())
snapshots.append((0, text))

print("Denoising trajectory:")
for t_val, text in snapshots:
    print(f"  t={t_val:3d}: '{text}'")

## Key Takeaways

1. The D3PM denoiser is a standard transformer conditioned on the timestep.
2. Training uses a simple cross-entropy loss predicting x_0 from x_t.
3. Sampling runs the reverse chain from all-mask to clean text.
4. Temperature controls the diversity-quality tradeoff.

## What's Next

In Lesson 04, we learn about **MDLM**, which simplifies discrete diffusion by using only masking transitions and a streamlined loss.